# 💄 Aesthetics-Aware Makeup Transfer — Exploration Notebook

Use this notebook to:
1. Visualise the face parser output on a sample image
2. Inspect the expert system's palette recommendations
3. Preview the makeup renderer output
4. Verify the 5-channel model input tensor

In [ ]:
import sys; sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.pipeline.face_parser  import FaceParser
from src.pipeline.expert_system import ExpertSystem
from src.pipeline.renderer      import MakeupRenderer
from src.utils.visualization    import draw_landmarks, draw_masks, save_pair_grid

print('Imports OK')

## 1. Load a test image

In [ ]:
# Replace with any FFHQ image path
IMG_PATH = '../data/ffhq/00001.png'

bgr = cv2.imread(IMG_PATH)
if bgr is None:
    # Create a synthetic face-coloured placeholder for demo
    bgr = np.full((256, 256, 3), [140, 170, 200], dtype=np.uint8)
    print('⚠  No image found — using placeholder')

bgr = cv2.resize(bgr, (256, 256))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(4,4))
plt.imshow(rgb); plt.title('Source Image'); plt.axis('off'); plt.show()

## 2. Face Parser — Landmarks & Masks

In [ ]:
parser = FaceParser(min_confidence=0.5)
parse  = parser.parse(bgr)

if parse is None:
    print('No face detected')
else:
    print(f'Face shape  : {parse.face_shape}')
    print(f'Skin RGB    : {parse.skin_rgb.astype(int)}')
    print(f'Bbox        : {parse.bbox}')
    print(f'Masks found : {list(parse.masks.keys())}')

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(rgb); axes[0].set_title('Original'); axes[0].axis('off')
    lm_img = draw_landmarks(bgr, parse)
    axes[1].imshow(cv2.cvtColor(lm_img, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Landmarks'); axes[1].axis('off')
    mask_img = draw_masks(bgr, parse)
    axes[2].imshow(cv2.cvtColor(mask_img, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Zone Masks'); axes[2].axis('off')
    plt.tight_layout(); plt.show()

## 3. Expert System — Palette Recommendation

In [ ]:
if parse:
    es       = ExpertSystem()
    analysis = es.analyze(parse.skin_rgb, parse.face_shape)
    plan     = analysis['plan']

    print(f"Undertone     : {analysis['undertone']}")
    print(f"Lightness     : {analysis['lightness']}")
    print(f"Face shape    : {analysis['face_shape']}")
    print(f"Lip color     : RGB{plan.lip_color}")
    print(f"Blush color   : RGB{plan.blush_color}")
    print(f"Contour zones : {plan.contour_zones}")
    print(f"Eyeshadows    : {plan.eyeshadow_colors}")

    # Show colour swatches
    colors = {
        'Lip'      : plan.lip_color,
        'Blush'    : plan.blush_color,
        'Contour'  : plan.contour_color,
        'Highlight': plan.highlight_color,
        'Eye 1'    : plan.eyeshadow_colors[0],
        'Eye 2'    : plan.eyeshadow_colors[1],
    }
    fig, axes = plt.subplots(1, len(colors), figsize=(12, 2))
    for ax, (name, rgb_c) in zip(axes, colors.items()):
        swatch = np.full((60, 60, 3), rgb_c, dtype=np.uint8)
        ax.imshow(swatch); ax.set_title(name, fontsize=9); ax.axis('off')
    plt.suptitle('Recommended Palette', fontsize=11)
    plt.tight_layout(); plt.show()

## 4. Renderer — Synthetic Makeup Application

In [ ]:
if parse:
    renderer = MakeupRenderer(blur_kernel=15, use_seamless=False)
    bgr_y    = renderer.render(bgr, parse, plan)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('X  (original)'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(bgr_y, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Y  (synthetic makeup)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

## 5. Model Architecture Summary

In [ ]:
from src.models.generator     import MaskConditionedUNet
from src.models.discriminator import PatchGANDiscriminator

G = MaskConditionedUNet()
D = PatchGANDiscriminator()

g_params = sum(p.numel() for p in G.parameters()) / 1e6
d_params = sum(p.numel() for p in D.parameters()) / 1e6
print(f'Generator      : {g_params:.2f}M parameters')
print(f'Discriminator  : {d_params:.2f}M parameters')

# Forward pass test
x     = torch.randn(1, 5, 256, 256)
inten = torch.tensor([1.0])
with torch.no_grad():
    out = G(x, inten)
print(f'G output shape : {out.shape}  range [{out.min():.2f}, {out.max():.2f}]')

## 6. Dataset Item Inspection

In [ ]:
from src.training.dataset import MakeupPairDataset

# Adjust path to your generated dataset
DATA_DIR = '../data/synthetic'

try:
    ds   = MakeupPairDataset(DATA_DIR, split='train', image_size=256)
    item = ds[0]
    print(f'Dataset size : {len(ds)}')
    print(f'x shape      : {item["x"].shape}  — channels: RGB + lip_mask + skin_mask')
    print(f'y shape      : {item["y"].shape}')
    print(f'intensity    : {item["intensity"]}')

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    channel_names = ['R', 'G', 'B', 'Lip Mask', 'Skin Mask']
    for i, name in enumerate(channel_names):
        ch = item['x'][i].numpy()
        axes[i].imshow(ch, cmap='gray' if i >= 3 else None)
        axes[i].set_title(name); axes[i].axis('off')
    plt.suptitle('5-Channel Input Tensor (x)')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'Dataset not ready yet: {e}')
    print('Run scripts/generate_dataset.py first.')